##Data Processing with Databricks Apache Spark (PySpark)

#### Download the entry point to programming the Data set with Spark.

In [0]:
from pyspark.sql import SparkSession

#### Import the python packages needed to connect to/extract the dataset

In [0]:
import pandas as pd
import requests

# Fetch the data.
df = pd.read_csv("https://ourworldindata.org/grapher/unemployment-rate.csv?v=1&csvType=full&useColumnShortNames=false", storage_options = {'User-Agent': 'Our World In Data data fetch/1.0'})

# Fetch the metadata
metadata = requests.get("https://ourworldindata.org/grapher/unemployment-rate.metadata.json?v=1&csvType=full&useColumnShortNames=false").json()

In [0]:
#  convert pandas dataframe into a spark dataframe
df_spark = spark.createDataFrame(df)


In [0]:
#checking schema 
df_spark.printSchema()

root
 |-- Entity: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- Year: long (nullable = true)
 |-- Unemployment, total (% of total labor force) (modeled ILO estimate): double (nullable = true)



In [0]:
#confirming if any rows of the dataset maybe dispropotionately larger than the rest. This is to plan for\
  #optimization techniques like salting incase the join key's data is skewed this way.
  
df_spark.groupBy("Entity").count().orderBy("count", ascending=False).show()


+-----------+-----+
|     Entity|count|
+-----------+-----+
|Afghanistan|   33|
|    Algeria|   33|
|  Argentina|   33|
|     Angola|   33|
|    Belgium|   33|
|    Albania|   33|
|      Benin|   33|
|    Bahamas|   33|
|    Belarus|   33|
|    Burundi|   33|
|    Bolivia|   33|
| Bangladesh|   33|
|   Barbados|   33|
|     Bhutan|   33|
| Azerbaijan|   33|
|    Armenia|   33|
|     Brunei|   33|
|     Belize|   33|
|     Brazil|   33|
|   Botswana|   33|
+-----------+-----+
only showing top 20 rows



In [0]:
#making column names fit for purpose
df_spark = df_spark.withColumnRenamed('Unemployment, total (% of total labor force) (modeled ILO estimate)','Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate')


In [0]:
#write raw unprocessed data to bronze layer
df_spark.write.format("delta").mode("overwrite").save("dbfs:/delta/bronze/unemployment")


In [0]:
#filter data 
df_spark = df_spark.filter(df_spark.Year>2012)


In [0]:
#transform column values by rounding to two decimal places
from pyspark.sql.functions import round
df_spark = df_spark.withColumn("Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate", round(df_spark["Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate"], 2))

In [0]:
#write to silver layer
df_spark.write.format("delta").mode("overwrite").save("dbfs:/delta/silver/unemployment")

In [0]:
#aggregate data
from pyspark.sql.functions import col, round

df_grouped = df_spark.groupBy("Entity") \
    .avg("Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate") \
    .withColumn("Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate", round(
        col("avg(Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate)"), 2)) \
    .drop("avg(Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate)")


In [0]:
#column name renaming
df_grouped = df_grouped.withColumnRenamed('Unemployment_total_percentage_of_labour_force_modeled_ILO_estimate','Average_Unemployment_total_percentage_2013-2023')

In [0]:
#writing aggregated data to gold layer with error handling 
import logging

try:
    df_grouped.write.format("delta").mode("overwrite").save("dbfs:/delta/gold/unemployment")
except Exception as e:
    logging.error(f"write failed: {e}")



In [0]:
# list the files paths present
%fs ls /delta


path,name,size,modificationTime
dbfs:/delta/bronze/,bronze/,0,0
dbfs:/delta/gold/,gold/,0,0
dbfs:/delta/silver/,silver/,0,0
